# Проект: книжные рекомендации

**Задача рекомендательной системы** — персонализировать рекомендации книг на основе поведения пользователей: их предпочтений по жанрам, дочитанным или дослушанным произведениям, факту прочтения, времени потребления контента, платформе использования и геолокации.

### Задача проекта:

-  построить ETL-пайплайн, который будет обрабатывать данные в пакетном режиме для работы рекомендательной системы.

Детали:

- ETL-пайплайн должен собирать **пять витрин** данных разной степени детализации для рекомендательной системы

### Описание витрин данных

#### user_content_mart

Решаемая задача:

- Обучение моделей рекомендаций разных типов (ranking, implicit feedback) на основе того, что пользователь уже читал или слушал.
- Расчёт признаков, основанных на взаимодействии пользователя с разными произведениями.


#### user_genre_mart

Решаемая задача:

- Построение профиля интересов пользователя.
- Рекомендации при «холодном старте» пользователя (англ. cold start scenario), то есть при отсутствии значительной истории данных.
- Генерация рекомендаций по жанрам.


#### user_mart

Решаемая задача:

- Хранение пользовательских фич, то есть определение профиля пользователя.
- Сегментация пользователей.


#### content_mart

Решаемая задача:

- Хранение контентных фич, то есть данных о популярности и качестве контента.
- Оценка популярности конкретных произведений.
- Создание сбалансированных и персонализированных рекомендаций, а не просто сводки популярного контента.


#### daily_user_mart

Решаемая задача:

- Определение динамики активности пользователей.
- Хранение признаков, основанных на временных паттернах.

### Исходные данные:

две таблицы в формате Parquet:

- <https://code.s3.yandex.net/data-scientist/dsplus/audition.parquet>,
- <https://code.s3.yandex.net/data-scientist/dsplus/content.parquet>

Ссылки для pyspark:

- s3a://s3-ds-source/audition.parquet
- s3a://s3-ds-source/content.parquet
- endpoint: <https://storage.yandexcloud.net>

In [13]:
# импорты
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from dotenv import load_dotenv
from pyspark.sql.functions import col, sum, isnull

load_dotenv()

True

### Создание Spark-сессии и чтение данных

In [2]:
def create_spark_session():
  """Функция создания Spark-сессии"""
  spark = (
      SparkSession.builder
        .appName('book_recommendations')
        .master("local[*]")
        .config(
            "spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4",
        )
        # endpoint
        .config("spark.hadoop.fs.s3a.endpoint", "https://storage.yandexcloud.net")
        # Ключ доступа
        .config("spark.hadoop.fs.s3a.access.key", os.environ["S3_ACCESS_KEY"])

        # Секретный ключ
        .config("spark.hadoop.fs.s3a.secret.key", os.environ["S3_SECRET_KEY"])

        # Реализация файловой системы для работы с S3 через протокол S3A
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

        # Используем path-style access:
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.executor.memory", "2g")
        .config("spark.executor.cores", "2")
        .config("spark.executor.instances", "2")
        .config("spark.driver.cores", "4")
        .config("spark.driver.memory", "8g")
        .config("spark.sql.shuffle.partitions", "32")
        .config("spark.network.timeout", "300")
        .getOrCreate()
  )
  return spark



In [3]:
spark = create_spark_session()

:: loading settings :: url = jar:file:/Users/a.plahotniuc/Workspace/practicum_ds/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/a.plahotniuc/.ivy2/cache
The jars for the packages stored in: /Users/a.plahotniuc/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fbf4d42f-9cfc-4f04-88a5-78168d92cafd;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 115ms :: artifacts dl 6ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-------------------

### Первичное знакомство с данными

In [4]:
def show_data_params(df):
    print('-'*30)
    
    print('Кол-во строк: ', df.count())
    print('Кол-во колонок: ', len(df.columns))
    df.printSchema()
    df.show(5, truncate=False)

In [5]:
# загрузка
df_audition = spark.read.parquet('s3a://s3-ds-source/audition.parquet')
df_content = spark.read.parquet('s3a://s3-ds-source/content.parquet')

26/11/03 12:13:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [6]:
# информация о датафреймах
show_data_params(df=df_audition)
show_data_params(df=df_content)

------------------------------
Кол-во строк:  1002896
Кол-во колонок:  12
root
 |-- audition_id: long (nullable = true)
 |-- puid: string (nullable = true)
 |-- usage_platform_ru: string (nullable = true)
 |-- msk_business_dt_str: string (nullable = true)
 |-- app_version: string (nullable = true)
 |-- adult_content_flg: boolean (nullable = true)
 |-- hours: double (nullable = true)
 |-- hours_sessions_long: double (nullable = true)
 |-- kids_content_flg: boolean (nullable = true)
 |-- main_content_id: string (nullable = true)
 |-- usage_geo_id: string (nullable = true)
 |-- __index_level_0__: long (nullable = true)



+-----------+------------------------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+-----------------------+-----------------+
|audition_id|puid                                |usage_platform_ru|msk_business_dt_str|app_version|adult_content_flg|hours             |hours_sessions_long|kids_content_flg|main_content_id|usage_geo_id           |__index_level_0__|
+-----------+------------------------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+-----------------------+-----------------+
|0          |68296628-f9d6-11ef-be00-c2c9fa6fd3d5|Станция          |2024-11-26         |NULL       |false            |0.0377777777777777|0.0377777777777777 |true            |oCURrBKV       |Алматы, Казахстан      |162              |
|1          |682966dc-f9d6-11ef-be00-c2c9fa6fd3d5|Станция          |

+---------------+-----------------+---------------------------------------------------------------------------------------+---------------------------+--------------------------------------------------------------+-----------------------+
|main_content_id|main_content_type|main_content_name                                                                      |main_content_duration_hours|published_topic_title_list                                    |main_author_id         |
+---------------+-----------------+---------------------------------------------------------------------------------------+---------------------------+--------------------------------------------------------------+-----------------------+
|A00hxVIL       |Book             |Давай поговорим о твоих доходах и расходах                                             |4.151                      |'Синхронизировано', 'Бизнес', 'Саморазвитие', 'Личные финансы'|Карл Ричардс           |
|A03if3j5       |Comicbook        |Минимализ

#### Анализ:

**Датафрейм audition**:
- Кол-во строк:  1002896, Кол-во колонок:  12

**Датафрейм content**:
- Кол-во строк:  31668, Кол-во колонок:  6


Общяя информация:
- все поля nullable
- явных ошибок в типах данных нет, данные соответствуют описанию, датасеты загружен корректно



## Предобработка и анализ данных

- Преобразуйте строковые даты msk_business_dt_str в формат timestamp для анализа временных трендов.
- Проверьте данные на наличие пропусков и аномалий.
- Примеры возможных аномалий: отрицательная длительность сессий или длительность сессий более 24 часов.
- Устраните все аномалии и заполните пропуски.

---
- Объедините таблицы audition и content по ключу main_content_id.После объединения проверьте, не продублировались ли строки в финальном датафрейме.
- Отфильтруйте данные по периоду от start_date до end_date и по местоположению usage_geo_id. Обе даты должны быть включены в выборку.
- Преобразуйте строковые значения из колонки published_topic_title_list в списки. Для деления можно использовать функцию split().
- Примените filter, select, withColumn, чтобы достичь трёх целей:
	- Получите распределение пользователей по платформам, используя данные из usage_platform_ru. Дополнительно вы также можете реализовать возможность добавлять колонки с распределением по другим признакам через аргументы запуска.
	- Проанализируйте популярность типов контента main_content_type и жанров published_topic_title_list.
	- Исследуйте зависимость длительности сессий hours от дня недели и геолокации usage_geo_id.

In [57]:
# преобразование колонки с датой
df_audition = df_audition.withColumn("msk_business_dt", F.to_timestamp(F.col("msk_business_dt_str"), "yyyy-MM-dd"))

### Исследование выбросов, пропусков и аномалий

In [58]:
def df_null_count_show(df):
    """Подсчет Null для каждой колонки датафрейма"""
    null_counts = df.select([
		F.sum(
			F.when(F.col(c).isNull(), 1).otherwise(0)
		).alias(c) for c in df.columns
	])
    print('Всего строк: ', df.count())
    print("Кол-во Null:")
    null_counts.show()


def df_nan_count_show(df):
    """Подсчет NaN для каждой колонки датафрейма"""

	# проверка работает не со всеми типами данных
    numeric_cols = [name for name, dtype in df.dtypes if dtype in ['float', 'double']]
    null_counts = df.select([
		F.sum(
			F.when(F.isnan(c), 1).otherwise(0)
		).alias(c) for c in numeric_cols
	])
    print('Всего строк: ', df.count())
    print("Кол-во NaN:")
    null_counts.show()
	

In [40]:
df_null_count_show(df=df_audition)
df_nan_count_show(df=df_audition)

df_null_count_show(df=df_content)
df_nan_count_show(df=df_content)


Всего строк:  1002896
Кол-во Null:


+-----------+----+-----------------+-------------------+-----------+-----------------+-----+-------------------+----------------+---------------+------------+-----------------+---------------+
|audition_id|puid|usage_platform_ru|msk_business_dt_str|app_version|adult_content_flg|hours|hours_sessions_long|kids_content_flg|main_content_id|usage_geo_id|__index_level_0__|msk_business_dt|
+-----------+----+-----------------+-------------------+-----------+-----------------+-----+-------------------+----------------+---------------+------------+-----------------+---------------+
|          0|   0|                0|                  0|     702511|                0|    0|                  0|               0|              0|           0|                0|              0|
+-----------+----+-----------------+-------------------+-----------+-----------------+-----+-------------------+----------------+---------------+------------+-----------------+---------------+

Всего строк:  1002896
Кол-во NaN:


+-----+-------------------+
|hours|hours_sessions_long|
+-----+-------------------+
|    0|                  0|
+-----+-------------------+

Всего строк:  31668
Кол-во Null:


+---------------+-----------------+-----------------+---------------------------+--------------------------+--------------+
|main_content_id|main_content_type|main_content_name|main_content_duration_hours|published_topic_title_list|main_author_id|
+---------------+-----------------+-----------------+---------------------------+--------------------------+--------------+
|              0|                0|                0|                          6|                         0|             0|
+---------------+-----------------+-----------------+---------------------------+--------------------------+--------------+

Всего строк:  31668
Кол-во NaN:
+---------------------------+
|main_content_duration_hours|
+---------------------------+
|                          0|
+---------------------------+



Анализ:
- `df_audition.app_version` - высокая доля 70% пропусков Null. Пропуски стоит оставить для дальнейшей замены уже в ходе пайплайна обучения
- `df_content.main_content_duration_hours` - несущественное кол-во Null пропусков (менее 1%). Подходит удаление малочисленных пропусков

### Исследование распределений

In [54]:
def show_stats(df, col_names: list[str]):
    """Сводная статистика и квантили"""
    df.select(*col_names).describe().show()
    
    col_names_num = [name for name, dtype in df.dtypes if dtype in ['float', 'double']]
    qnt = [0.05, 0.25, 0.5, 0.75, 0.95]
    rel_error = 0.01
    quantiles_col_info = df.approxQuantile(col_names_num, qnt, rel_error)
    print(f"Квантили распределений для колонок: {col_names_num}")
    print(quantiles_col_info)


In [55]:
show_stats(df=df_audition, col_names=df_audition.columns)

+-------+------------------+--------------------+-----------------+-------------------+--------------------+------------------+-------------------+---------------+--------------------+-----------------+
|summary|       audition_id|                puid|usage_platform_ru|msk_business_dt_str|         app_version|             hours|hours_sessions_long|main_content_id|        usage_geo_id|__index_level_0__|
+-------+------------------+--------------------+-----------------+-------------------+--------------------+------------------+-------------------+---------------+--------------------+-----------------+
|  count|           1002896|             1002896|          1002896|            1002896|              300385|           1002896|            1002896|        1002896|             1002896|          1002896|
|   mean| 688420.2683468675|                NULL|             NULL|               NULL| 7.758675294207244E7|0.4194248205040685|0.41561910149156744|           NULL|                NULL| 843

Квантили распределений для колонок: ['hours', 'hours_sessions_long']
[[0.0030555555555555, 0.0405555555555555, 0.1247222222222222, 0.23, 1.8976], [0.0, 0.0430555555555555, 0.1261111111111111, 0.2262909090909091, 1.824722222222222]]


In [56]:
show_stats(df=df_content, col_names=df_content.columns)

+-------+---------------+-----------------+--------------------+---------------------------+--------------------------+--------------------+
|summary|main_content_id|main_content_type|   main_content_name|main_content_duration_hours|published_topic_title_list|      main_author_id|
+-------+---------------+-----------------+--------------------+---------------------------+--------------------------+--------------------+
|  count|          31668|            31668|               31668|                      31662|                     31668|               31668|
|   mean|           NULL|             NULL|  1939.4655172413793|          8.315491295691517|        0.6693055555555555|  2.9800925925925923|
| stddev|           NULL|             NULL|   421.4754680844953|          7.966225047175883|         0.916685374388227|  1.5572836218257042|
|    min|       A00hxVIL|        Audiobook|            #Elysium|                        0.0|                   'Audio'|'Классика', 'Худо...|
|    max|    

Анализ:
- `hours`, `hours_sessions_long`, `main_content_duration_hours` имеют сильные выбросы. Масимум отличается от p95 на порядки. Имеет смысл применить фильтрацию выбросов
- длительность сессии более 24 часов считается аномалией, стоит провести фильтрацию 

### Фильтрация данных

In [ ]:
# время в формате строки больше не нужно
df_audition = df_audition.drop("msk_business_dt_str")

In [60]:
# пропуски в данных
df_content = df_content.filter(col("main_content_duration_hours").isNotNull())


In [62]:
# фильтрация аномалий
df_content = df_content.filter(col("main_content_duration_hours") <= 24)
df_audition = df_audition.filter((col("hours") <= 24) & (col("hours_sessions_long") <= 24))